# Visualize Best Agent

This notebook loads a population checkpoint (`latest.pkl`), extracts the best agent from it, displays its program code, and visualizes it playing FlappyBird with human rendering.


In [4]:
# Import required libraries
import flappy_bird_env  # noqa
import numpy as np
import os
import pickle
from pathlib import Path
from scipy.special import expit

# Disable headless mode for human rendering
os.environ.pop('SDL_VIDEODRIVER', None)

try:
    import pygame
    pygame.init()
    print("✓ Pygame initialized for human rendering")
except Exception as e:
    print(f"Warning: Could not initialize pygame: {e}")

import gymnasium as gym
from memory_system import MemoryConfig, MemoryType
from evaluator import FlappyBirdEvaluator, FlappyBirdEvaluatorConfig


✓ Pygame initialized for human rendering


## Load Best Agent from Population Checkpoint

Load the checkpoint file (`latest.pkl`) which contains a population, then extract the best agent from it.


In [5]:
# Path to checkpoint file (contains a population)
checkpoint_path = "gen_0854.pkl"

# Import EvolutionEngine to load checkpoint
from evolution_engine import EvolutionEngine

# Check if file exists
if not Path(checkpoint_path).exists():
    print(f"❌ Checkpoint file not found: {checkpoint_path}")
    print("Please update checkpoint_path to point to your checkpoint pickle file.")
    best_agent = None
    generation = None
    population = None
else:
    print(f"Loading checkpoint from: {checkpoint_path}")
    
    # Load the checkpoint (contains population and metadata)
    checkpoint = EvolutionEngine.load_checkpoint(checkpoint_path)
    population = checkpoint['population']
    generation = checkpoint.get('generation', 'unknown')
    checkpoint_fitness = checkpoint.get('fitness', None)
    
    print(f"✓ Checkpoint loaded successfully!")
    print(f"  Generation: {generation}")
    if checkpoint_fitness is not None:
        print(f"  Best fitness in checkpoint: {checkpoint_fitness:.4f}")
    print(f"  Population size: {len(population)}")
    
    # Get the best agent from the population
    # Prefer best_ever if available (best agent across all generations)
    if population.best_ever is not None:
        best_agent = population.best_ever
        best_generation = population.best_ever_generation
        print(f"  Using best_ever agent (found at generation {best_generation})")
    else:
        best_agent = population.get_best()
        print(f"  Using best agent from current generation")
    
    print(f"\n✓ Best agent extracted!")
    print(f"  Agent ID: {best_agent.id}")
    if best_agent.fitness is not None:
        print(f"  Fitness: {best_agent.fitness:.4f}")
    else:
        print(f"  Fitness: not set")
    print(f"  Program length: {len(best_agent.program)}")
    print(f"  Age: {best_agent.age}")
    if best_agent.parent_ids:
        print(f"  Parent IDs: {best_agent.parent_ids}")


Loading checkpoint from: gen_0854.pkl
✓ Checkpoint loaded successfully!
  Generation: 854
  Best fitness in checkpoint: 1.4000
  Population size: 1000
  Using best_ever agent (found at generation 854)

✓ Best agent extracted!
  Agent ID: 427756
  Fitness: 1.4000
  Program length: 22
  Age: 0
  Parent IDs: (349344, 326529)


## Display Program Code

Show the full program and the effective program (with introns removed).


In [6]:
# Reconstruct MemoryConfig from the agent's MemoryBank
# This is needed for instruction string formatting
memory = best_agent.memory
memory_cfg = MemoryConfig(
    n_scalar=8,
    n_vector=8,
    n_matrix=8,
    n_obs_scalar=0,
    n_obs_vector=1,
    n_obs_matrix=1,
    vector_size=64,
    matrix_shape=(64,64),
)

# Output registers (default for FlappyBird, adjust if your config uses different)
output_registers = [(MemoryType.SCALAR, 0)]

print("="*80)
print("FULL PROGRAM")
print("="*80)
print()

for i, instr in enumerate(best_agent.program.instructions):
    print(f"{i:4d}: {instr.to_resolved_str(memory_cfg)}")

print()
print("="*80)
print(f"Total: {len(best_agent.program)} instructions")
print("="*80)


FULL PROGRAM

   0: scalar[0→0] = scalar_add(obs_matrix[0→0][21,24], scalar[2480→0])
   1: scalar[7→7] = scalar_div_protected(scalar[6241→1], obs_matrix[0→0][11,48])
   2: scalar[7→7] = scalar_conditional(scalar[7446→6], scalar[3251→3])
   3: scalar[4→4] = scalar_conditional(scalar[3319→7], scalar[2399→7])
   4: scalar[1→1] = automl_scalar_exp(scalar[5567→7])
   5: scalar[7→7] = scalar_conditional(scalar[714→2], scalar[4458→2])
   6: scalar[7→7] = automl_scalar_cos(obs_matrix[0→0][44,30])
   7: scalar[0→0] = scalar_conditional(scalar[8512→0], scalar[9003→3])
   8: scalar[4→4] = scalar_conditional(scalar[7780→4], scalar[6702→6])
   9: scalar[2→2] = automl_scalar_cos(scalar[8374→6])
  10: scalar[6→6] = automl_scalar_log(scalar[1179→3])
  11: scalar[7→7] = scalar_sub(obs_matrix[0→0][62,15], obs_matrix[0→0][31,40])
  12: scalar[2→2] = automl_scalar_log(obs_matrix[0→0][10,53])
  13: scalar[6→6] = automl_scalar_exp(obs_matrix[0→0][23,22])
  14: scalar[0→0] = scalar_div_protected(scalar[5266→

In [7]:
# Show effective program (with introns removed)
effective_program = best_agent.get_effective_program(output_registers)

print("="*80)
print("EFFECTIVE PROGRAM (INTRONS REMOVED)")
print("="*80)
print()

if len(effective_program.instructions) > 0:
    for i, instr in enumerate(effective_program.instructions):
        print(f"{i:4d}: {instr.to_resolved_str(memory_cfg)}")
else:
    print("  (No effective instructions found)")

print()
print("="*80)
print(f"Effective: {len(effective_program)} instructions (out of {len(best_agent.program)} total)")
if len(best_agent.program) > 0:
    effective_ratio = len(effective_program) / len(best_agent.program)
    print(f"Effective code rate: {effective_ratio:.3f} ({effective_ratio*100:.1f}%)")
print("="*80)


EFFECTIVE PROGRAM (INTRONS REMOVED)

   0: scalar[0→0] = scalar_div_protected(scalar[5266→2], scalar[336→0])

Effective: 1 instructions (out of 22 total)
Effective code rate: 0.045 (4.5%)


## Create Evaluator with Human Rendering

Set up the FlappyBird evaluator with human rendering mode to visualize the agent playing.


In [8]:
# Get the expected vector size from the agent's memory (for feature_vector strategy)
feature_vector_size = best_agent.memory.vector_size
print(f"Agent's vector size: {feature_vector_size}")
print(f"Agent's matrix shape: {best_agent.memory.matrix_shape}")

# Create evaluator config for visualization
# Use the same config as training, but with human rendering
evaluator_config = FlappyBirdEvaluatorConfig(
    env_id="FlappyBird-v0",
    episodes=3,  # Run 3 episodes to see the agent play
    max_steps=500,
    output_register=0,
    render_mode="human",  # Human rendering to see the game
    rng_seed=42,
    patch_strategy="feature_vector",  # Using feature_vector strategy
    color_channel=2,  # Not used for feature_vector, but required
    normalize=True,
    quantization_factor=0.5,  # Not used for feature_vector
    feature_vector_size=feature_vector_size,  # Match agent's vector size
    output_registers=[(MemoryType.SCALAR, 0)],
    n_jobs=1,  # Sequential for visualization
)

print("\nCreating FlappyBird evaluator with human rendering...")
evaluator = FlappyBirdEvaluator(config=evaluator_config)
print("✓ Evaluator created!")
print(f"  Episodes: {evaluator.episodes}")
print(f"  Max steps per episode: {evaluator.max_steps}")
print(f"  Render mode: {evaluator.config.render_mode}")
print(f"  Patch strategy: feature_vector")
print(f"  Feature vector size: {feature_vector_size}")
print()
print("⚠️  NOTE: FlappyBird windows will appear when you run the next cell!")
print("   Close the windows or press Ctrl+C to stop.")


Agent's vector size: 22
Agent's matrix shape: (22, 22)

Creating FlappyBird evaluator with human rendering...
✓ Evaluator created!
  Episodes: 3
  Max steps per episode: 500
  Render mode: human
  Patch strategy: feature_vector
  Feature vector size: 22

⚠️  NOTE: FlappyBird windows will appear when you run the next cell!
   Close the windows or press Ctrl+C to stop.


## Visualize Agent Playing

Run the agent and watch it play FlappyBird. The game windows will appear showing the agent's performance.


In [9]:
# Run the agent and visualize
print("="*80)
print("RUNNING BEST AGENT")
print("="*80)
print()

total_reward = 0.0
episode_rewards = []

for episode_idx in range(evaluator.episodes):
    print(f"\nEpisode {episode_idx + 1}/{evaluator.episodes}")
    print("-" * 80)
    
    # Seed the episode
    episode_seed = int((evaluator.config.rng_seed + episode_idx * 100) % (2**31))
    observation, _ = evaluator.env.reset(seed=episode_seed)
    observation = np.asarray(observation, dtype=np.float32)
    
    # Copy memory for this episode
    memory = best_agent.memory.copy()
    episode_reward = 0.0
    steps = 0
    
    for step in range(evaluator.max_steps):
        # Process observation
        # Note: _process_observation returns different formats based on strategy:
        # - feature_vector: returns dict {'vector': [...], 'matrix': [...]}
        # - quantized/full_image: returns tuple (observations_list, 'matrix')
        result = evaluator._process_observation(observation)
        
        # Load observations into memory
        if isinstance(result, dict):
            # feature_vector strategy returns both vector and matrix
            memory.load_observation(result)
        else:
            # quantized/full_image strategy returns tuple
            processed_observations, obs_type = result
            if obs_type == 'vector':
                memory.load_observation({'vector': processed_observations})
            else:
                memory.load_observation({'matrix': processed_observations})
        
        # Execute the program
        best_agent.program.execute(memory,debug=True)
        
        # Read action from output register
        action_value = memory.read_scalar(evaluator.output_register)
        normalized = expit(action_value)  # Sigmoid
        action = 1 if normalized >= 0.5 else 0
        
        # Take step in environment
        observation, reward, terminated, truncated, _ = evaluator.env.step(action)
        observation = np.asarray(observation, dtype=np.float32)
        episode_reward += reward
        steps += 1
        
        if terminated or truncated:
            break
    
    episode_rewards.append(episode_reward)
    total_reward += episode_reward
    
    print(f"  Steps: {steps}")
    print(f"  Reward: {episode_reward:.2f}")
    print(f"  Action value (scalar[0]): {action_value:.4f}")
    print(f"  Normalized (sigmoid): {normalized:.4f}")
    print(f"  Action chosen: {'FLAP' if action == 1 else 'NOOP'}")

print()
print("="*80)
print("SUMMARY")
print("="*80)
print(f"Total episodes: {evaluator.episodes}")
print(f"Average reward: {total_reward / evaluator.episodes:.2f}")
print(f"Rewards per episode: {[f'{r:.2f}' for r in episode_rewards]}")
print("="*80)

# Close the evaluator
evaluator.close()
print("\n✓ Visualization complete!")


RUNNING BEST AGENT


Episode 1/3
--------------------------------------------------------------------------------
Executing instruction 0: scalar[0] = scalar_add(obs_matrix[0]→scalar[5464], scalar[2480])
Executing instruction 1: scalar[7] = scalar_div_protected(scalar[6241], obs_matrix[0]→scalar[4848])
Executing instruction 2: scalar[7] = scalar_conditional(scalar[7446], scalar[3251])
Executing instruction 3: scalar[4] = scalar_conditional(scalar[3319], scalar[2399])
Executing instruction 4: scalar[1] = automl_scalar_exp(scalar[5567])
Executing instruction 5: scalar[7] = scalar_conditional(scalar[714], scalar[4458])
Executing instruction 6: scalar[7] = automl_scalar_cos(obs_matrix[0]→scalar[2846])
Executing instruction 7: scalar[0] = scalar_conditional(scalar[8512], scalar[9003])
Executing instruction 8: scalar[4] = scalar_conditional(scalar[7780], scalar[6702])
Executing instruction 9: scalar[2] = automl_scalar_cos(scalar[8374])
Executing instruction 10: scalar[6] = automl_scalar_log(

## Additional Information

Display additional details about the best agent's memory and constants.


In [10]:
# Display memory information
print("="*80)
print("BEST AGENT MEMORY INFORMATION")
print("="*80)
print()

memory = best_agent.memory

print("Scalar registers (working):")
for i in range(min(8, memory.n_scalar)):
    print(f"  scalar[{i}]: {memory.scalars[i]:.6f}")

print()
print("Vector registers (working):")
for i in range(min(3, memory.n_vector)):
    vec_str = ", ".join([f"{v:.3f}" for v in memory.vectors[i][:5]])
    if len(memory.vectors[i]) > 5:
        vec_str += "..."
    print(f"  vector[{i}]: [{vec_str}]")

print()
print("Matrix registers (working):")
for i in range(min(3, memory.n_matrix)):
    print(f"  matrix[{i}]: shape {memory.matrices[i].shape}, "
          f"mean={memory.matrices[i].mean():.4f}, "
          f"std={memory.matrices[i].std():.4f}")

print()
print("Observation registers:")
if memory.n_obs_matrix > 0:
    print(f"  obs_matrix[0]: shape {memory.obs_matrices[0].shape}")

print("="*80)


BEST AGENT MEMORY INFORMATION

Scalar registers (working):
  scalar[0]: -15.986492
  scalar[1]: 1.462659
  scalar[2]: -6.806849
  scalar[3]: -6.864674
  scalar[4]: -5.528834
  scalar[5]: -2.011588
  scalar[6]: -5.057519
  scalar[7]: -2.668893

Vector registers (working):
  vector[0]: [2.836, 5.818, 0.424, -0.690, 0.022...]
  vector[1]: [0.520, -0.132, 2.336, -0.588, 0.285...]
  vector[2]: [-0.198, 4.815, -0.752, 0.522, 0.285...]

Matrix registers (working):
  matrix[0]: shape (22, 22), mean=0.0260, std=2.1665
  matrix[1]: shape (22, 22), mean=-0.0034, std=1.9576
  matrix[2]: shape (22, 22), mean=-0.0096, std=1.8345

Observation registers:
  obs_matrix[0]: shape (22, 22)
